In [8]:
import win32com.client as com
import os
import pandas as pd
import geopandas as gpd
from shapely import wkt

In [9]:
folder = r'C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara'
visum_network = os.path.join(folder, "Red Base GDL", "RedBase 120826", "RedBase 300726 - conectores_final.ver")

Visum = com.Dispatch("Visum.Visum")
Visum.LoadVersion(visum_network)
C = com.constants

In [21]:
connectors = pd.DataFrame({
    "ZoneNo": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues("ZoneNo")],
    "NodeNo": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues("NodeNo")],
    "Direction": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues("Direction")],
    "Length": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues("Length")],
    "CreatedBy": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues("CREATED_BY")],
    "highway": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues(r"Node\Distinct:InLinks\HIGHWAY")],
    "clave_ageb": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues(r"Zone\CLAVE_AGEB")],
    "geometry": [i[1] for i in Visum.Net.Connectors.GetMultiAttValues("WKTPolyWGS84")],
})

connectors["geometry"] = connectors["geometry"].apply(wkt.loads)
connectors = gpd.GeoDataFrame(connectors, geometry="geometry", crs="EPSG:4326")

In [22]:
connectors = connectors[['ZoneNo','clave_ageb', 'NodeNo', 'Direction', 'Length', 'highway', 'CreatedBy', 'geometry']]
connectors = connectors.rename(columns={'NodeNo':'node_id', 'Length':'length', 'CreatedBy':'created_by'})
connectors['length'] = connectors['length']*1000
#connectors = connectors.drop_duplicates(subset=['clave_ageb'], keep='first')

In [23]:
connectors

,ZoneNo,clave_ageb,node_id,Direction,length,highway,created_by,geometry
0,1.0,1403900010026,133628.0,1.0,496.287583,"primary,primary_link,residential",algorithm,"LINESTRING (-103.3838 20.7084, -103.3883 20.7098)"
1,1.0,1403900010026,133628.0,2.0,496.287583,"primary,primary_link,residential",algorithm,"LINESTRING (-103.3883 20.7098, -103.3838 20.7084)"
2,1.0,1403900010026,160984.0,1.0,599.026140,"residential,secondary",algorithm,"LINESTRING (-103.3838 20.7084, -103.3781 20.7085)"
3,1.0,1403900010026,160984.0,2.0,599.026140,"residential,secondary",algorithm,"LINESTRING (-103.3781 20.7085, -103.3838 20.7084)"
4,1.0,1403900010026,161064.0,1.0,251.850471,"['living_street', 'footway'],residential",algorithm,"LINESTRING (-103.3838 20.7084, -103.3822 20.7067)"
...,...,...,...,...,...,...,...,...
19319,9053.0,1412401760601,9432.0,2.0,84.711997,"residential,unclassified",algorithm,"LINESTRING (-103.0794 20.5255, -103.0786 20.5255)"
19320,9053.0,1412401760601,29441.0,1.0,75.104067,"residential,service",algorithm,"LINESTRING (-103.0786 20.5255, -103.0781 20.526)"
19321,9053.0,1412401760601,29441.0,2.0,75.104067,"residential,service",algorithm,"LINESTRING (-103.0781 20.526, -103.0786 20.5255)"
19322,9054.0,141240368,36128.0,1.0,1648.868254,residential,visum_fallback,"LINESTRING (-103.0136 20.7346, -103.0283 20.74)"


In [25]:
visum_objects = "Visum Objects"
connectors.to_file(os.path.join(folder, visum_objects, "Connectors", "connectors.shp"))

In [26]:
path=os.path.join(folder, visum_objects, "Connectors", "connectors.gpkg")
connectors.to_file(path, layer="connectors", driver="GPKG")